# This copied file uses q2 for line-by-line differential atmospheric-parameter analysis
### Replace `ispec.model_spectrum_from_ew()` for parameter iteration
### About q2
- q2 is designed for standard 1D/LTE EW spectroscopic analysis using MOOG 2019 (silent), and is especially good at:
    - Solving Teff/logg/[Fe/H]/vt (microturbulence) through Fe I/Fe II excitation and ionization balance
    - Performing strict line-by-line differential analysis by specifying a reference star (usually the Sun)
- q2 takes two CSV inputs:
    - `stars.csv`: initial values for each star (`id`, Teff, logg, [Fe/H], vt)
    - `lines.csv`: atomic parameters and EW for each line (lambda, species, EP, loggf, EW), with EW measured beforehand

## Notebook workflow
- Import module functions (cell 1)
- Input preparation (cell 3)
- `find_linemasks` (cell 4)
- Filter and save linemasks (cell 5)
- FePlot (cell 7, now via a shared utility function)
- Prepare manual line-removal table + `line_review` (cell 9)
- Apply deleted-line filtering to the atomic linelist (cell 11)
- Apply quick LineQA filtering (cell 13)
- Replace manual EW-writing function definition with module function usage (cell 14)
- Fe abundance (cell 17)
- Fe trend plots (cell 19)
- q2 fitting (cell 22)
- Save q2 dump (cell 24)
- Export q2 CSV (cell 25)

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import logging
import multiprocessing
from multiprocessing import Pool
import matplotlib.pyplot as plt
from scipy.stats import norm
import shutil
import q2

from inlist import target, PIPELINE_OPTIONS_OVERRIDE
from run_context import apply_context_globals
from atmos_lbl_pipeline import (
    step_log,
    load_or_prepare_inputs,
    run_find_linemasks_part1,
    filter_and_save_linemasks,
    plot_gaussian_fits_by_element,
    prepare_manual_line_edit_tables,
    apply_deleted_lines_to_atomic_linelist,
    fast_apply_lineqa_to_linemasks,
    set_line_ew_in_linemask,
    compute_fe_abundance,
    plot_fe_abundance_trends,
    run_q2_fit_and_export_inputs,
    save_q2_dump,
    export_q2_dump_csv,
)

ctx = apply_context_globals(
    globals(),
    target=target,
    pipeline_options_override=PIPELINE_OPTIONS_OVERRIDE,
    prepare_io=True,
)

# --- Part I switch: keep in sync with PIPELINE_OPTIONS['run_find_linemasks'] ---
RUN_FIND_LINEMASKS = PIPELINE_OPTIONS.get("run_find_linemasks", True)

# --- Bedell Table1 external EW: keep in sync with PIPELINE_OPTIONS['input_has_external_ews'] ---
USE_EXTERNAL_EW = PIPELINE_OPTIONS.get("input_has_external_ews", False)
_bedell_q2_tar_ew = None
_bedell_q2_sun_ew = None

# Part I spectral line finding: force skip when external EW is used
# (including convolution / find_linemasks / plotting, etc.)
RUN_PART1_SPECTRUM = RUN_FIND_LINEMASKS and (not USE_EXTERNAL_EW)


MOOGSILENT is available in /Users/jiayue/q2-tools/MOOG-for-q2


**Bedell Table1 external EW**: when `userlist.PIPELINE_OPTIONS['input_has_external_ews']=True`, the notebook follows the `USE_EXTERNAL_EW` branch and skips Part I spectrum-based EW measurement. Outputs are written to `output/withEW_Bedell/<target>/`. A standard Part I `*_melendez2014_star_fitted_linemasks.txt` file must already exist as the template. By default, **Fe lines keep pipeline-template values only** (`bedell_use_pipeline_fe_only`), while **non-Fe** lines with Table1 EWs **overwrite matching template rows, and missing template lines are appended into fitted linemasks** (`bedell_merge_missing_table1_lines`) to avoid line loss.

# Part I: inspect Gaussian fits of Fe lines before atmospheric-parameter iteration
### `example.py`: `find_linemasks()`

**When only recomputing atmospheric parameters (q2)**: set `"run_find_linemasks": False` in `PIPELINE_OPTIONS` inside `userlist.py`, then run this notebook. It will skip convolution, `find_linemasks`, plotting, manual line-table preparation, and the second find step, and will load `star_linemasks` from the existing `.../linemasks/<target>_melendez2014_star_fitted_linemasks.txt`. Then continue as usual from **Part II (Fe)** or **Part III / q2**. (If you only need q2, you may additionally skip the MOOG step in Part II cell 17 and run only cells 21-22. Cell 17 recomputes MOOG abundances and is relatively slow.)

In [2]:
step_log("1", "Prepare spectrum and linelist inputs.")
_inputs = load_or_prepare_inputs(
    ispec=ispec,
    target=target,
    output_folder=output_folder,
    output_linelist_path=output_linelist_path,
    linelist_target_path=linelist_target_path,
    spectrum_norm_path=spectrum_norm_path,
    ispec_dir=ispec_dir,
    row_target=row_target,
    PIPELINE_OPTIONS=PIPELINE_OPTIONS,
    RUN_PART1_SPECTRUM=RUN_PART1_SPECTRUM,
    run_find_linemasks=RUN_FIND_LINEMASKS,
)

star_spectrum = _inputs["star_spectrum"]
atomic_linelist = _inputs["atomic_linelist"]
atomic_linelist_file = _inputs["atomic_linelist_file"]
star_linemasks = _inputs["star_linemasks"]
iron_star_linemasks = _inputs["iron_star_linemasks"]
recover_star_linemasks = _inputs["recover_star_linemasks"]
linemasks = _inputs["linemasks"]
_bedell_q2_tar_ew = _inputs["bedell_q2_tar_ew"]
_bedell_q2_sun_ew = _inputs["bedell_q2_sun_ew"]


[STEP 1] Prepare spectrum and linelist inputs.
[STEP 1] External EW mode enabled (Bedell Table1).
[bedell_table1_ew] use_pipeline_fe_only=True：Fe 1/Fe II 的 EW 与线表均保留管线模板，非 Fe 使用 Table1（覆盖 + 必要时追加）。
[bedell_table1_ew] template=/Users/jiayue/iSpec/output/MegantestSample/HIP14614/linemasks/HIP14614_melendez2014_star_fitted_linemasks.txt
[bedell_table1_ew] wrote fitted -> /Users/jiayue/iSpec/output/withEW_Bedell/HIP14614/linemasks/HIP14614_melendez2014_star_fitted_linemasks.txt
[bedell_table1_ew] EW overwrite hits: 105 (on original 228 template rows); appended from Table1: 56; final line count: 284
[bedell_table1_ew] table EW<=0 skipped in overwrite: 0; Fe rows skipped in overwrite pass: 78; append skipped (Discard species): 0; unique table keys used for non-Fe overwrite map: 339
[bedell_table1_ew] q2：use_pipeline_fe_only=True，Fe EW 不读取 Table1，完全使用管线 linemasks 中的 Fe。
[STEP 1] Prepared external-EW linemasks: raw=284, Fe=78, filtered=284


In [3]:
if RUN_PART1_SPECTRUM:
    star_linemasks = run_find_linemasks_part1(
        ispec=ispec,
        star_spectrum=star_spectrum,
        atomic_linelist=atomic_linelist,
        from_resolution=from_resolution,
        to_resolution=to_resolution,
        min_depth=0.05,
        max_depth=1.00,
    )

In [4]:
if RUN_PART1_SPECTRUM:
    _saved = filter_and_save_linemasks(
        ispec=ispec,
        target=target,
        output_folder=output_folder,
        star_linemasks=star_linemasks,
        ew_min=10,
        ew_max=100,
    )
    star_linemasks = _saved["star_linemasks"]
    iron_star_linemasks = _saved["iron_star_linemasks"]
    recover_star_linemasks = _saved["recover_star_linemasks"]
    linemask_output_folder = _saved["linemask_output_folder"]



- make the plot

In [5]:
if RUN_PART1_SPECTRUM:
    _plot_stats = plot_gaussian_fits_by_element(
        star_spectrum=star_spectrum,
        linemasks=star_linemasks,
        output_dir=output_folder + "/figs_Fe_GaussianFits",
        element_filter=["Fe 1", "Fe 2"],
        prefix="FeFit",
        w_range=0.25,
        progress_every=10,
    )

[STEP 4][0/91] Start gaussian plotting for 91 lines.
[STEP 4][10/91] plotting in progress: saved=10, skipped=0
[STEP 4][20/91] plotting in progress: saved=20, skipped=0
[STEP 4][30/91] plotting in progress: saved=30, skipped=0
[STEP 4][40/91] plotting in progress: saved=40, skipped=0
[STEP 4][50/91] plotting in progress: saved=50, skipped=0
[STEP 4][60/91] plotting in progress: saved=60, skipped=0
[STEP 4][70/91] plotting in progress: saved=70, skipped=0
[STEP 4][80/91] plotting in progress: saved=80, skipped=0
[STEP 4][90/91] plotting in progress: saved=90, skipped=0
[STEP 4][91/91] plotting in progress: saved=91, skipped=0
[STEP 4] Gaussian plotting completed. saved=91, skipped=0


### Create a table for manually recording deleted lines

In [5]:
if RUN_PART1_SPECTRUM:
    _qa_paths = prepare_manual_line_edit_tables(
        output_folder=output_folder,
        target=target,
        output_linelist_path=output_linelist_path,
        PIPELINE_OPTIONS=PIPELINE_OPTIONS,
    )
    deleted_file_path = _qa_paths["deleted_file_path"]
    modified_file_path = _qa_paths["modified_file_path"]
    step_log(
        "5",
        f"Manual edit tables ready: deleted={deleted_file_path}, modified={modified_file_path}",
    )

### Manually remove poorly fitted Fe lines (and lines of other elements)
- When editing `Lines_deleted.tsv` for the second time (for other elements), you may notice extra tab characters in `linelist_for_target.tsv`. In that case, use `linemasks/linelist_for_target_copied.tsv` as the copy-paste source for the second edit.

In [4]:
if RUN_PART1_SPECTRUM:
    filtered_linelist = apply_deleted_lines_to_atomic_linelist(
        ispec=ispec,
        output_folder=output_folder,
        linelist_target_path=linelist_target_path,
    )
    atomic_linelist_file = linelist_target_path

[STEP 6] Applying deleted-line table to atomic linelist.
[STEP 6] Deleted 13 lines from atomic linelist.
[STEP 6] Filtered atomic linelist: 284 lines.


### Rewrite with updated line selections

In [5]:
if RUN_PART1_SPECTRUM:
    _fast_lq = fast_apply_lineqa_to_linemasks(
        ispec=ispec,
        target=target,
        output_folder=output_folder,
        star_linemasks=star_linemasks,
        ew_min=10,
        ew_max=100,
    )
    star_linemasks = _fast_lq["star_linemasks"]
    iron_star_linemasks = _fast_lq["iron_star_linemasks"]
    recover_star_linemasks = _fast_lq["recover_star_linemasks"]
    step_log("7", f"Fast LineQA kept lines: {len(star_linemasks)}")


[STEP 7] Applying LineQA deletions directly to fitted linemasks.
[STEP 7] Fast LineQA kept lines: 228


In [6]:
step_log("7", "Function set_line_ew_in_linemask is imported from atmos_lbl_pipeline.py")


[STEP 7] Function set_line_ew_in_linemask is imported from atmos_lbl_pipeline.py


In [9]:
lm_path = linemask_output_folder + f"/{target}_melendez2014_star_fitted_linemasks.txt"
df_ew_modify = pd.read_csv(output_folder+"linemasks/Lines_EW_modified.tsv", sep="\t")

use_A  = "wave_A"  in df_ew_modify.columns
use_nm = "wave_nm" in df_ew_modify.columns
if not (use_A or use_nm):
    raise ValueError("df_ew_modify must include either 'wave_A' or 'wave_nm'.")

n_ok, n_err = 0, 0
for k, r in df_ew_modify.iterrows():
    try:
        elem = str(r["element"]).strip()
        ew   = float(r["ew_mA_new"])
        if use_A:
            lam  = float(r["wave_A"])
            unit = "A"
        else:
            lam  = float(r["wave_nm"])
            unit = "nm"

        # Exact match by wavelength precision in the linemask.
        # Typical precision is ~0.001 A; loosen `tol` to 0.01 A or 0.001 nm if needed.
        set_line_ew_in_linemask(lm_path, element=elem, wave_value=lam,
            ew_mA=ew, wave_unit=unit, tol=0.0, note_tag="EW_man=%.1f mA")
        n_ok += 1
    except Exception as e:
        print(f"[Skipped] row {k}: {r.get('element','?')} lambda={r.get('wave_A', r.get('wave_nm','?'))} -> {e}")
        n_err += 1

print(f"Done: succeeded {n_ok}, failed {n_err}.")

完成：成功 0 条，失败 0 条。


# Part II: compute Fe I and Fe II abundances first
### `example.py`: `determine_abundances_from_ew()`
- Switch `code` from `"spectrum"` to `"moog"`.

### Tips
- Clearly document the models used in your paper:
    - Atmospheric model: `ispec_dir + "/input/atmospheres/MARCS.GES/"`
    - Solar abundance model: `ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"`
- Report mean and standard deviation for Fe I/H and Fe II/H.
    - Also report the number of lines used.
    - Check for outliers among these lines:
        - If sigma > 0.1, inspect carefully.
        - If sigma > 0.15, an outlier is very likely.
        - Continuum placement issues are also possible.
- It is helpful to inspect slopes first:
    - REW vs. [Fe/H], and low excitation potential vs. [Fe/H].

In [10]:
linemasks = ispec.read_line_regions(output_folder + f"/linemasks/{target}_melendez2014_star_fitted_linemasks.txt")
_fe_abund = compute_fe_abundance(
    ispec=ispec,
    linemasks=linemasks,
    ispec_dir=ispec_dir,
    initial_teff=initial_teff,
    initial_logg=initial_logg,
    initial_MH=initial_MH,
    initial_feh_err=row_target["[Fe/H]_err"],
)

linemasks = _fe_abund["linemasks"]
spec_abund = _fe_abund["spec_abund"]
normal_abund = _fe_abund["normal_abund"]
x_over_h = _fe_abund["x_over_h"]
x_over_fe = _fe_abund["x_over_fe"]
fe1_abund = _fe_abund["fe1_abund"]
fe2_abund = _fe_abund["fe2_abund"]
spec_abund_fe1 = _fe_abund["spec_abund_fe"]

model = _fe_abund["model"]
code = _fe_abund["code"]
teff = _fe_abund["teff"]
logg = _fe_abund["logg"]
MH = _fe_abund["MH"]
alpha = _fe_abund["alpha"]
microturbulence_vel = _fe_abund["microturbulence_vel"]
solar_abundances_file = _fe_abund["solar_abundances_file"]

initial_vmic = microturbulence_vel

[STEP 8] Computing Fe abundance with MOOG.
[STEP 8] [Fe/H] ref=-0.1090 +/- 0.004
[STEP 8] Fe I lines=69 mean=-0.1167 std=0.0624
[STEP 8] Fe II lines=9 mean=-0.1373 std=0.0471


In [13]:
# Find indices where [Fe II/H] equals a target value
target_abund = -0.125

fe2_mask = (linemasks['element'] == "Fe 1") & (~np.isnan(x_over_h))
fe2_indices = np.where(fe2_mask)[0]
target_idx = fe2_indices[np.isclose(x_over_h[fe2_mask], target_abund, atol=1e-4)]

# Print the corresponding spectral-line information
if len(target_idx) > 0:
    print(linemasks[target_idx][['wave_nm', 'element', 'ew', 'loggf']])
else:
    print("Didn't find the corresponding line.")

Didn't find the corresponding line.


In [14]:
plot_fe_abundance_trends(
    linemasks=linemasks,
    x_over_h=x_over_h,
    use_external_ew=USE_EXTERNAL_EW,
)

[STEP 9][0/4] Start Fe trend plotting.
[STEP 9][1/4] Rendered plot: [Fe I/H] vs. Excitation Potential
[STEP 9][2/4] Rendered plot: [Fe I/H] vs. log(REW)
[STEP 9][3/4] Rendered plot: [Fe II/H] vs. Excitation Potential
[STEP 9][4/4] Rendered plot: [Fe II/H] vs. log(REW)
[STEP 9] Fe trend plotting completed.


/Users/jiayue/iSpec/Code/atmos_lbl_pipeline.py:719: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Part III: Atmospheric parameter fitting

In [15]:
# Read spectrum and linelist
#lineregions_path = linemask_output_folder+"/"+target+"_melendez2014_star_fitted_linemasks.txt"
# lineregions_ = star_linemasks

#line_regions = ispec.read_line_regions(ispec_dir + lineregions_path.format(code="width"))
#print(f"Successfully read {len(line_regions)} lines from the line regions.")

#--- Read the Normalized spectrum -------------------------------------------------------------
spectrum_path = spectrum_norm_path
normalized_star_spectrum = ispec.read_spectrum(spectrum_path)

from abund_plot import *
initial_vmic = ispec.estimate_vmic(initial_teff, initial_logg, initial_MH)

In [19]:
_q2 = run_q2_fit_and_export_inputs(
    ispec=ispec,
    linemasks=linemasks,
    target=target,
    output_folder=output_folder,
    ispec_dir=ispec_dir,
    solar_label=solar_label,
    solar_lines_csv=solar_lines_csv,
    instrument_name=instrument_name,
    from_resolution=from_resolution,
    initial_teff=initial_teff,
    initial_logg=initial_logg,
    initial_MH=initial_MH,
    initial_vmic=initial_vmic,
    PIPELINE_OPTIONS=PIPELINE_OPTIONS,
    bedell_q2_tar_ew=globals().get("_bedell_q2_tar_ew"),
    bedell_q2_sun_ew=globals().get("_bedell_q2_sun_ew"),
)
linemasks_fe = _q2["linemasks_fe"]
q2_dir = _q2["q2_dir"]
solution_csv = _q2["solution_csv"]
row = _q2["row"]
teff_q2 = _q2["teff_q2"]
logg_q2 = _q2["logg_q2"]
feh_q2 = _q2["feh_q2"]
vt_q2 = _q2["vt_q2"]
# use step_log to print the reference parameters
step_log("10", f"Reference parameters: Teff = {initial_teff}, logg = {initial_logg}, [Fe/H] = {initial_MH}")


[STEP 10] Preparing q2 inputs and running atmospheric fit.
[export_q2_inputs] target Fe lines: 78
[export_q2_inputs] solar resolution requested=115000.0, used=115000
[export_q2_inputs] sun Fe lines   : 80
[export_q2_inputs] common Fe lines: 74
[export_q2_inputs] lines.csv rows : 74 -> /Users/jiayue/iSpec/output/MegantestSample/HIP14614/q2_work/lines_q2.csv
------------------------------------------------------
Initializing ...
- Date and time: 28-May-2026, 17:35:23
- Model atmospheres: marcs
- Star data: stars_q2.csv
- Line list: lines_q2.csv
------------------------------------------------------

***
Sun
***
Reference star. No calculations needed.
it Teff logg [Fe/H]  vt           [Fe/H]
-- ---- ---- ------ ----      --------------
 0 5772 4.44  0.000 1.00 --->  0.000+/-0.000
--
------------------------------------------------------
   D[Fe/H]    ||    D[Fe/H] Fe I   |   D[Fe/H] Fe II
 0.000  0.000 ||  0.000  0.000   0 |  0.000  0.000   0
----------------------------------------------

# Saving

In [22]:
_dump = save_q2_dump(
    ispec=ispec,
    output_atmos_params_dumpfile_path=output_atmos_params_dumpfile_path,
    solution_row=row,
    linemasks_fe=linemasks_fe,
    target=target,
    solution_csv=solution_csv,
)
dump_file_q2 = _dump["dump_file_q2"]
params_q2 = _dump["params_q2"]
errors_q2 = _dump["errors_q2"]

csv_atmos_q2 = export_q2_dump_csv(
    ispec=ispec,
    target=target,
    output_atmos_params_dumpfile_path=output_atmos_params_dumpfile_path,
)


[STEP 11] Saving q2 payload to iSpec-style dump.
[STEP 11] Saved q2 dump: /Users/jiayue/iSpec/output/MegantestSample/HIP14614/atmos_params_HIP14614_q2_q2.dump
[STEP 12] Wrote q2 atmos CSV: /Users/jiayue/iSpec/output/MegantestSample/HIP14614/atmos_params_HIP14614_q2_q2.csv
